# Intelligent Fault Diagnosis Tutorial

For the 2026 E-Rotors course

_By the Aalto University ARotor Laboratory (Aleksanteri Hämäläinen & Aku Karhinen)_


In [3]:
## Do not edit this cell

# await __import__("piplite").install("openconmo", deps=False)

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy

## Test cell figures


![Test Figure](figures/karhi.png)


In [150]:
def import_polito(
    classes=["H", "IR", "OR"],
    rpms=[523, 937],
    radial_forces=[62.4, 124.8],
    axial_forces=[0, 49],
    verbose=False,
):
    """
    Import the Polito dataset and return the relevant data as a dictionary.

    Description of the dataset can be found here: https://data.niaid.nih.gov/resources?id=zenodo_13913253
    Article with more details about the dataset can be found here: https://www.mdpi.com/1424-8220/23/1/211


    """

    class_map = {"H": "Healthy", "IR": "InnerRaceDamage", "OR": "OuterRaceDamage"}
    class_map_inverse = {v: k for k, v in class_map.items()}

    mat_files = (Path("data") / "PolitoBearingFaultData").glob("**/*.mat")
    mat_files = filter(lambda f: class_map_inverse[f.parent.name] in classes, mat_files)
    mat_files = filter(
        lambda f: int(f.name.split("_")[0].replace("rpm", "")) in rpms, mat_files
    )
    mat_files = filter(
        lambda f: float(f.name.split("_")[1].replace("kN", "")) in radial_forces,
        mat_files,
    )
    mat_files = filter(
        lambda f: float(f.name.split("_")[2].replace("kN.mat", "")) in axial_forces,
        mat_files,
    )

    dataset = {}

    for f in mat_files:
        if verbose:
            print(f)
        data = scipy.io.loadmat(f)

        for i in range(5):
            try:
                signal = data[f"Signal_{i}"]
            except KeyError:
                continue

            unit = signal[0, 0]["y_values"]["quantity"][0, 0]["label"][0, 0][0]
            # print("acceleration unit:", unit)

            if unit == "g":
                # Signal not actually in g, but m/s^2, but since we are fine with m/s^2, wecan ignore the scaling factor
                acc_signal = signal[0, 0]["y_values"]["values"][0, 0][
                    :, 3
                ]  # Last bearing (4th) is the one with the fault
                break

            # elif unit == "rpm"
            #     print()

        rpm, radial_load, axial_load = (
            f.stem.replace("rpm", "").replace("kN", "").split("_")
        )

        # FIXME: Add class label
        class_label = class_map_inverse[f.parent.name]
        dataset[(class_label, int(rpm), float(radial_load), float(axial_load))] = (
            acc_signal
        )

    return dataset

In [154]:
polito_dataset = import_polito()
polito_dataset

{('OR',
  937,
  62.4,
  0.0): array([-0.18921762, -0.34718829,  0.94137625, ...,  0.53888582,
         1.26343472, -1.11406109]),
 ('OR',
  937,
  124.8,
  0.0): array([-0.02827105, -0.35553734, -1.4893551 , ..., -1.52134602,
        -0.3013925 , -1.86704636]),
 ('OR',
  523,
  124.8,
  0.0): array([ 1.46629187, -0.67875311,  0.14606707, ..., -0.26204449,
        -1.70428118, -0.31850392]),
 ('OR',
  523,
  124.8,
  49.0): array([-0.99080925,  0.64461292,  0.77092333, ...,  2.42973938,
        -0.40546632, -1.81124824]),
 ('OR',
  523,
  62.4,
  0.0): array([ 2.69922362, -0.68065437, -0.1423472 , ...,  0.62179769,
        -0.0384387 ,  0.48639426]),
 ('OR',
  937,
  124.8,
  49.0): array([-0.81919903, -0.70355227, -0.1670637 , ...,  2.31367929,
         0.28122251,  0.47490398]),
 ('IR',
  937,
  62.4,
  0.0): array([-0.22476309, -1.60483654, -1.49067772, ...,  0.8586297 ,
         1.34188274, -0.47382935]),
 ('IR',
  937,
  124.8,
  0.0): array([-1.14059619, -0.83969968, -0.46382702,

In [ ]:
def signal_windowing(signal, window_size, overlap=0.9):
    """
    Split a signal into overlapping windows.

    Parameters
    ----------
    signal : np.ndarray
        The input signal to be windowed.
    window_size : int
        The size of each window (number of samples).
    overlap : int
        The number of samples that overlap between consecutive windows.

    Returns
    -------
    np.ndarray
        An array of shape (num_windows, window_size) containing the windowed segments of the input signal.
    """

    step = max(1, int(window_size * (1 - overlap)))
    num_windows = 1 + (len(signal) - window_size) // step
    windows = np.array(
        [signal[i * step : i * step + window_size] for i in range(num_windows)]
    )

    return windows


def polito_to_sklearn_format(
    polito_dict,
    train_radial_loads=[],
    test_radial_loads=[],
    train_axial_loads=[],
    test_axial_loads=[],
    train_rpms=[],
    test_rpms=[],
):
    """
    Convert the Polito dataset dictionary into NumPy arrays suitable for scikit-learn.

    Samples are added to the training and/or test sets based on the provided
    filter lists. Any filter list left empty means "accept all" for that field.
    A sample can appear in both sets if train and test filters overlap.

    Class labels are mapped as follows:
    - "H" -> 0
    - "IR" -> 1
    - "OR" -> 2

    Parameters
    ----------
    polito_dict : dict
        Dataset dictionary produced by `import_polito`.
    train_radial_loads : list, optional
        Radial loads for training samples.
    test_radial_loads : list, optional
        Radial loads for test samples.
    train_axial_loads : list, optional
        Axial loads for training samples.
    test_axial_loads : list, optional
        Axial loads for test samples.
    train_rpms : list, optional
        RPM values for training samples.
    test_rpms : list, optional
        RPM values for test samples.

    Returns
    -------
    tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]
        `(train_X, train_y, test_X, test_y)` where:
        - `train_X` and `test_X` contain vibration signals
        - `train_y` and `test_y` contain class labels
    """

    train_X = []
    train_y = []
    test_X = []
    test_y = []

    class_label_map = {"H": 0, "IR": 1, "OR": 2}

    for (class_label, rpm, radial_load, axial_load), signal in polito_dict.items():
        # Check which filters are specified (non-empty)
        train_rpm_check = not train_rpms or rpm in train_rpms
        train_radial_check = not train_radial_loads or radial_load in train_radial_loads
        train_axial_check = not train_axial_loads or axial_load in train_axial_loads

        test_rpm_check = not test_rpms or rpm in test_rpms
        test_radial_check = not test_radial_loads or radial_load in test_radial_loads
        test_axial_check = not test_axial_loads or axial_load in test_axial_loads

        windowed_signal = None
        if train_rpm_check and train_radial_check and train_axial_check:
            # FIXME: Window size
            windowed_signal = signal_windowing(signal, window_size=2048, overlap=0.9)

            train_X.append(windowed_signal)
            train_y.extend(
                [class_label_map[class_label]] * len(windowed_signal)
            )  # Extend with the same label for each window

        # `if` instead of `elif` because a sample can be in both sets if filters overlap
        if test_rpm_check and test_radial_check and test_axial_check:
            if windowed_signal is None:
                windowed_signal = signal_windowing(
                    signal, window_size=2048, overlap=0.9
                )

            test_X.append(windowed_signal)
            test_y.extend([class_label_map[class_label]] * len(windowed_signal))

    print(train_X[0].shape)
    print(train_X[1].shape)

    train_X = np.concatenate(train_X, axis=0)
    train_y = np.array(train_y)
    test_X = np.concatenate(test_X, axis=0)
    test_y = np.array(test_y)

    return train_X, train_y, test_X, test_y

In [205]:
train_X, train_y, test_X, test_y = polito_to_sklearn_format(
    polito_dataset, train_rpms=[523], test_rpms=[937]
)

(6014, 2048)
(6014, 2048)


In [204]:
test_X.shape

(45090, 2048)

In [198]:
train_y

array([2, 2, 2, ..., 0, 0, 0])